In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# folder containing datasets
start_time_folder = Path("dataset/Nancy_museum/start_obs_artworks")
end_time_folder = Path("dataset/Nancy_museum/end_obs_artworks")
artworks_df = pd.read_csv("dataset/Nancy_museum/artworks_dataset.csv")
visitors_information = pd.read_csv("dataset/Nancy_museum/post_questionnaire_formatted.csv")


def get_ds_from_folder(file_ext, folder_path):

    all_dfs = []
    
    for file in folder_path.glob(file_ext):
        
        df = pd.read_csv(file, decimal=",")
    
        # add dataset name as a new column, removes ".csv"
        df["user_id"] = file.stem
    
        # append to list
        all_dfs.append(df)

    combined_df = pd.concat(all_dfs, ignore_index=True)

    return combined_df
    
# combine all datasets
start_time_df = get_ds_from_folder('*.csv', start_time_folder)
end_time_df = get_ds_from_folder('*.csv', end_time_folder)

In [4]:
end_time_df["user_id"] = end_time_df["user_id"].str.replace("_end", "", regex=False)

# create occurrence indices
start_time_df["match_id"] = (start_time_df.groupby(["user_id", "paintingId"]).cumcount())

end_time_df["match_id"] = (end_time_df.groupby(["user_id", "paintingId"]).cumcount())

merged_df = pd.merge(start_time_df, end_time_df, on=["user_id", "paintingId", "match_id"], suffixes=("_start", "_end"))
merged_df[["timestamp_start", "timestamp_end"]] = merged_df[["timestamp_start", "timestamp_end"]].astype(float)

merged_df["duration_seconds"] = (merged_df["timestamp_end"] - merged_df["timestamp_start"])
merged_df["duration_seconds"] = (merged_df["duration_seconds"].round(2))

# remover items_ from samples in user_id column
merged_df["user_id"] = merged_df["user_id"].str.replace("items_", "", regex=False)

# add data about paintings
merged_df = pd.merge(merged_df, artworks_df[["image id", "painting name", "artist", "date"]], left_on="paintingId", right_on="image id", how="left")

# visitor_id and user_id are different
visitors_information["visitor_id"] = visitors_information["visitor_id"].astype(str)

merged_df = pd.merge(merged_df, visitors_information[["visitor_id", "art_knowledge", "art_involvement", "visit_rate"]], left_on="user_id", right_on="visitor_id", how="left")

merged_df = merged_df.drop(["match_id", "timestamp_start", "timestamp_end", "floorNumber_end", "image id", "visitor_id"], axis=1)
merged_df.rename(columns={'painting name': 'painting_name'}, inplace=True)

# replace user ids with user simplier representations
# create mapping
unique_users = merged_df["user_id"].unique()
user_map = {uid: f"User_{i+1}" for i, uid in enumerate(unique_users)}

# replace values
merged_df["user_id"] = merged_df["user_id"].map(user_map)

merged_df.to_csv("processed_data/pre_processed_ds.csv", index=False, encoding="utf-8-sig")